In [111]:
# Open .txt as read only file and print number of characters
with open('../data/the-verdict.txt', 'r', encoding="utf-8") as file:
    content = file.read()
    print(len(content))

20479


In [112]:
# Using re to split text into words, white spaces, punctuation characters
import re
text = 'Hello, world. This, is a test.'
re.split(r'(\s)', text)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']

In [113]:
re.split(r'([,.]|\s)', text)

['Hello',
 ',',
 '',
 ' ',
 'world',
 '.',
 '',
 ' ',
 'This',
 ',',
 '',
 ' ',
 'is',
 ' ',
 'a',
 ' ',
 'test',
 '.',
 '']

In [114]:
# To remove empty spaces
[item for item in re.split(r'([,.]|\s)', text) if item.strip()]

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']

In [115]:
text1 = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text1)
[item for item in result if item.strip()]

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']

In [116]:
#Tokenize raw text from the verdict

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', content)
preprocessed = [item for item in preprocessed if item.strip()]

len(preprocessed)

4690

In [117]:
preprocessed[:30]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius',
 '--',
 'though',
 'a',
 'good',
 'fellow',
 'enough',
 '--',
 'so',
 'it',
 'was',
 'no',
 'great',
 'surprise',
 'to',
 'me',
 'to',
 'hear',
 'that',
 ',',
 'in']

In [118]:
# Create a vocabulary and sort them alphabetically
vocab1 = {token:integer for integer, token in enumerate(sorted(set(preprocessed)))}
# [(key, value) for key, value in vocab1.items() if value <= 50]

In [119]:
len(vocab1)

1130

In [120]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_int = vocab
        self.int_str = {vocab[s]:s for s in vocab}
        
    def encode(self, text):
        results = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preproc_results = [item for item in results if item.strip()]
        encoded = [self.str_int[word] for word in preproc_results]
        return encoded
        
    def decode(self, tokens):
        decoded_words = [self.int_str[token] for token in tokens]
        text = " ".join(decoded_words)
        clean_text = re.sub(r'\s([,.:;?_!"()\'])', r'\1', text)
        return decoded_words

In [121]:
tokenizer = SimpleTokenizerV1(vocab1)
text = """"It's the last he painted, you know,"
Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [122]:
print(tokenizer.decode(ids))

['"', 'It', "'", 's', 'the', 'last', 'he', 'painted', ',', 'you', 'know', ',', '"', 'Mrs', '.', 'Gisburn', 'said', 'with', 'pardonable', 'pride', '.']


In [123]:
# Include <|unk|> and <|endoftext|>

new_preprocessed = sorted(set(preprocessed)) + ['<|endoftext|>', '<|unk|>']
vocab2 = {word:id for id, word in enumerate(new_preprocessed)}

In [124]:
len(vocab2)

1132

In [125]:
for k, v in vocab2.items():
    if v > len(vocab2) - 3:
        print(k, v)

<|endoftext|> 1130
<|unk|> 1131


In [131]:
# Tokenizer with <|unk|> and <|endoftext|> cases

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_int = vocab
        self.int_str = {i:s for s, i in vocab.items()}
        
    def encode(self, text):
        preproc = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preproc = [item.strip() for item in preproc if item.strip()]
        
        encoded = []
        for word in preproc:
            if word in self.str_int:
                encoded.append(self.str_int[word])
            else:
                encoded.append(self.str_int['<|unk|>'])
        # encoded.append(self.str_int['<|endoftext|>'])
        return encoded
    
    def decode(self, tokens):
        decoded_words = " ".join([self.int_str[token] for token in tokens])
        clean_text = re.sub(r'\s([,.:;?_!"()\'])', r'\1', decoded_words)
        return clean_text
        

Why end of text is not appended inside the encode loop?

In [132]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [133]:
# New tokenizer using vocab with unk and endoftext tokens
tokenizer_2 = SimpleTokenizerV2(vocab2)
print(tokenizer_2.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [134]:
print(tokenizer_2.decode(tokenizer_2.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.
